In [1]:
import pandas as pd

champion = pd.read_csv("../data/raw/ChampionTbl.csv")
summoner = pd.read_csv("../data/raw/SummonerMatchTbl.csv")

In [2]:
champion.head()

,ChampionId,ChampionName
0,0,No Champion
1,1,Annie
2,2,Olaf
3,3,Galio
4,4,TwistedFate


In [3]:
summoner.head()

,SummonerMatchId,SummonerFk,MatchFk,ChampionFk
0,1,1,EUW1_7565751492,902
1,2,1,EUW1_7565549583,902
2,3,1,EUW1_7564803077,16
3,4,1,EUW1_7564368646,103
4,5,1,EUW1_7564332041,800


In [4]:
merged = summoner.merge(
    champion,
    left_on="ChampionFk",
    right_on="ChampionId",
    how="left"
)

In [5]:
merged.head()

,SummonerMatchId,SummonerFk,MatchFk,ChampionFk,ChampionId,ChampionName
0,1,1,EUW1_7565751492,902,902,Milio
1,2,1,EUW1_7565549583,902,902,Milio
2,3,1,EUW1_7564803077,16,16,Soraka
3,4,1,EUW1_7564368646,103,103,Ahri
4,5,1,EUW1_7564332041,800,800,Mel


In [6]:
stats = pd.read_csv("../data/raw/MatchStatsTbl.csv")

In [7]:
final = merged.merge(
    stats,
    left_on="SummonerMatchId",
    right_on="SummonerMatchFk",
    how="left"
)

In [8]:
final.head()

,SummonerMatchId,SummonerFk,MatchFk,ChampionFk,ChampionId,ChampionName,MatchStatsId,SummonerMatchFk,MinionsKilled,DmgDealt,...,PrimarySlot3,SecondarySlot1,SecondarySlot2,SummonerSpell1,SummonerSpell2,CurrentMasteryPoints,EnemyChampionFk,DragonKills,BaronKills,visionScore
0,1,1,EUW1_7565751492,902,902,Milio,1,1,30,4765,...,8453,8345,8347,4,7,902,51,0,0,67
1,2,1,EUW1_7565549583,902,902,Milio,2,2,29,8821,...,8453,8345,8347,4,7,902,236,0,0,88
2,3,1,EUW1_7564803077,16,16,Soraka,3,3,34,6410,...,8237,8345,8347,4,7,16,498,0,0,97
3,4,1,EUW1_7564368646,103,103,Ahri,4,4,51,22206,...,8106,8226,8210,4,14,103,54,0,0,0
4,5,1,EUW1_7564332041,800,800,Mel,5,5,0,39106,...,0,0,0,2202,2201,800,12,0,0,0


In [9]:
enemy = champion.rename(
    columns={
        "ChampionId": "EnemyChampionFk",
        "ChampionName": "EnemyChampionName"
    }
)

In [10]:
final = final.merge(
    enemy,
    on="EnemyChampionFk",
    how="left"
)

In [11]:
final.head()

,SummonerMatchId,SummonerFk,MatchFk,ChampionFk,ChampionId,ChampionName,MatchStatsId,SummonerMatchFk,MinionsKilled,DmgDealt,...,SecondarySlot1,SecondarySlot2,SummonerSpell1,SummonerSpell2,CurrentMasteryPoints,EnemyChampionFk,DragonKills,BaronKills,visionScore,EnemyChampionName
0,1,1,EUW1_7565751492,902,902,Milio,1,1,30,4765,...,8345,8347,4,7,902,51,0,0,67,Caitlyn
1,2,1,EUW1_7565549583,902,902,Milio,2,2,29,8821,...,8345,8347,4,7,902,236,0,0,88,Lucian
2,3,1,EUW1_7564803077,16,16,Soraka,3,3,34,6410,...,8345,8347,4,7,16,498,0,0,97,Xayah
3,4,1,EUW1_7564368646,103,103,Ahri,4,4,51,22206,...,8226,8210,4,14,103,54,0,0,0,Malphite
4,5,1,EUW1_7564332041,800,800,Mel,5,5,0,39106,...,0,0,2202,2201,800,12,0,0,0,Alistar


In [13]:
final.isnull().sum()

SummonerMatchId              0
SummonerFk                   0
MatchFk                      0
ChampionFk                   0
ChampionId                   0
ChampionName                 0
MatchStatsId                 0
SummonerMatchFk              0
MinionsKilled                0
DmgDealt                     0
DmgTaken                     0
TurretDmgDealt               0
TotalGold                    0
Lane                    170385
Win                          0
item1                        0
item2                        0
item3                        0
item4                        0
item5                        0
item6                        0
kills                        0
deaths                       0
assists                      0
PrimaryKeyStone              0
PrimarySlot1                 0
PrimarySlot2                 0
PrimarySlot3                 0
SecondarySlot1               0
SecondarySlot2               0
SummonerSpell1               0
SummonerSpell2               0
CurrentM

In [14]:
final["Lane"].value_counts(dropna=False)

Lane
NaN        170385
BOTTOM     123826
MIDDLE     104366
TOP        101192
JUNGLE      98385
UTILITY     58623
NONE        51041
SUPPORT        14
Name: count, dtype: int64

In [15]:
final.shape

(707832, 38)

In [16]:
stats.shape

(732308, 31)

In [17]:
stats["Lane"].isnull().sum()

np.int64(177309)

In [19]:
stats["Lane"].value_counts(dropna=False)

Lane
NaN        177309
BOTTOM     128258
MIDDLE     107962
TOP        104169
JUNGLE     101660
UTILITY     60501
NONE        52435
SUPPORT        14
Name: count, dtype: int64

유틸리티=서폿에 병합

In [20]:
final["Lane"] = final["Lane"].replace({
    "SUPPORT": "UTILITY"
})

In [21]:
lane_df = final[
    final["Lane"].isin([
        "TOP",
        "JUNGLE",
        "MIDDLE",
        "BOTTOM",
        "UTILITY"
    ])
].copy()

In [22]:
lane_df["Lane"].value_counts()

Lane
BOTTOM     123826
MIDDLE     104366
TOP        101192
JUNGLE      98385
UTILITY     58637
Name: count, dtype: int64

In [23]:
final.to_csv("../data/processed/final_dataset.csv", index=False)

lane_df.to_csv("../data/processed/lane_dataset.csv", index=False)